# ALE3DCNN Best Recipe + Global Context Variants

Runs only the current best atlas-free CNN recipe and deeper/global-context variants. The frozen text-projection side variant lives in `ale_3dcnn_frozen_text_variant_colab.ipynb`.


In [ ]:
import os, sys, time, json, platform
from pathlib import Path
print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch check failed:', exc)


In [ ]:
!pip -q install nilearn nibabel huggingface-hub safetensors adapters transformers pyarrow matplotlib pandas scikit-learn tqdm umap-learn
REPO_URL = os.environ.get('NEUROVLM_REPO_URL', 'https://github.com/neurovlm/neurovlm.git')
REPO_BRANCH = os.environ.get('NEUROVLM_REPO_BRANCH', 'neurovlm_gnn')
REPO_DIR = os.environ.get('NEUROVLM_REPO_DIR', '/content/neurovlm_gnn')
if not os.path.exists(REPO_DIR):
    !git clone --branch $REPO_BRANCH --single-branch $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin $REPO_BRANCH
    !git -C $REPO_DIR checkout $REPO_BRANCH
    !git -C $REPO_DIR pull --ff-only origin $REPO_BRANCH
os.chdir(REPO_DIR)
!pip -q install -e '.[viz,notebook,metrics]'
sys.path.insert(0, str(Path(REPO_DIR) / 'src'))
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

DRIVE_ROOT = '/content/drive/MyDrive/neurovlm'
RUNS_DIR = f'{DRIVE_ROOT}/runs_ale_3dcnn_best_global'
CACHE_DIR = f'{DRIVE_ROOT}/data_ale_3dcnn/ale_caches'
LOCAL_CACHE_DIR = '/content/ale_caches_ale_3dcnn'
EVAL_RESOURCE_DIR = '/content/drive/MyDrive/neurovlm_evaluation_resources'
RUN_STAMP = time.strftime('%Y%m%d_%H%M%S')
for d in [RUNS_DIR, CACHE_DIR, LOCAL_CACHE_DIR, EVAL_RESOURCE_DIR]:
    os.makedirs(d, exist_ok=True)

# Use the improved sparse autoencoder checkpoint for matching architecture.
# For the original plain CNN best recipe, this can point to:
# /content/drive/MyDrive/neurovlm/runs_ale_3dcnn_autoencoder_pretrain/autoencoder_atlas_free_20260510_194641/checkpoints/best_cnn_autoencoder.pt
AUTOENCODER_CHECKPOINT = os.environ.get('ALE_CNN_AE_CHECKPOINT', '')
CACHE_FILE = f'{LOCAL_CACHE_DIR}/atlas_free_ale_4mm_fwhm9p0_crop_float16.pt'
COMPARISON_FILE = f'{RUNS_DIR}/best_global_context_comparison.csv'
print('AUTOENCODER_CHECKPOINT:', AUTOENCODER_CHECKPOINT or '<set this before running variants>')


In [ ]:
BASE_ARGS = [
    '--mode', 'atlas_free',
    '--epochs', '150',
    '--batch-size-auto',
    '--lr-cnn', '1e-4', '--lr-proj', '1e-5',
    '--warmup-epochs', '5', '--temperature', '0.07',
    '--val-interval', '5', '--early-stopping-patience', '25',
    '--out-dim', '384', '--dropout', '0.1', '--norm', 'group',
    '--kernel-fwhm-mm', '9', '--resolution-mm', '4', '--cache-dtype', 'float16',
    '--cache-file', CACHE_FILE,
    '--encoder-init', 'autoencoder_pretrained', '--autoencoder-checkpoint', AUTOENCODER_CHECKPOINT,
    '--text-proj-init', 'pretrained_infonce',
    '--semantic-eval', '--eval-resource-dir', EVAL_RESOURCE_DIR,
    '--train-sanity-n', '512', '--num-workers', '4',
    '--comparison-file', COMPARISON_FILE,
]

VARIANTS = [
    dict(name='ae_pretrained_finetune_cnn_pretrained_text_trainable_plain', model='ale_3dcnn', base_channels=48, num_blocks=4, extra=[]),
    dict(name='ae_pretrained_finetune_resnet48_multiscale_attention', model='ale_3dcnn_resnet', base_channels=48, num_blocks=4, extra=['--multi-scale', '--global-context', 'attention']),
    dict(name='ae_pretrained_finetune_resnet64_multiscale_attention', model='ale_3dcnn_resnet', base_channels=64, num_blocks=4, extra=['--multi-scale', '--global-context', 'attention']),
    dict(name='ae_pretrained_finetune_resnet48_dilated_multiscale_attention', model='ale_3dcnn_resnet', base_channels=48, num_blocks=4, extra=['--use-dilation', '--multi-scale', '--global-context', 'attention']),
    dict(name='ae_pretrained_finetune_resnet48_5stage_multiscale_se', model='ale_3dcnn_resnet', base_channels=48, num_blocks=5, extra=['--multi-scale', '--global-context', 'se']),
]

def run_variant(v):
    if not AUTOENCODER_CHECKPOINT:
        raise ValueError('Set AUTOENCODER_CHECKPOINT or ALE_CNN_AE_CHECKPOINT before training.')
    run_dir = f'{RUNS_DIR}/{v["name"]}_{RUN_STAMP}'
    args = BASE_ARGS + [
        '--model', v['model'],
        '--base-channels', str(v['base_channels']),
        '--num-blocks', str(v['num_blocks']),
        '--run-dir', run_dir,
        '--checkpoint-dir', f'{run_dir}/checkpoints',
    ] + v['extra']
    cmd = 'python experiments/train_ale_cnn.py ' + ' '.join(map(str, args))
    print(cmd)
    return os.system(cmd)


In [ ]:
# Run the best recipe first, then opt into the global-context variants.
# for variant in VARIANTS:
#     code = run_variant(variant)
#     if code != 0:
#         raise RuntimeError(f'Variant failed: {variant["name"]}')

run_variant(VARIANTS[0])


In [ ]:
# Uncomment when the base run finishes and the AE checkpoint matches the architecture.
# for variant in VARIANTS[1:]:
#     code = run_variant(variant)
#     if code != 0:
#         raise RuntimeError(f'Variant failed: {variant["name"]}')
